# تقطیر دانش ریاضی: معلم ← دانش‌آموز
**قبل از اجرا:** Runtime ← Change runtime type ← **T4 GPU**، سپس سلول‌ها را به ترتیب اجرا کن.

- معلم: `Qwen/Qwen2.5-Math-1.5B-Instruct`
- دانش‌آموز: `Qwen/Qwen2.5-0.5B` (مدل پایه، بدون تخصص ریاضی)
- روش: تقطیر در سطح دنباله (حل‌های معلم که جواب نهایی‌شان درست است، فیلتر و روی آن‌ها آموزش می‌دهیم)
- تست: ۱۰۰ مسئلهٔ ثابت از بخش test دیتاست GSM8K که هرگز در آموزش استفاده نمی‌شود
- همهٔ خروجی‌ها (مدل‌ها، داده، گزارش) در Google Drive ذخیره می‌شوند

In [ ]:
!pip -q install -U transformers datasets accelerate huggingface_hub

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, re, random, math, time, torch

BASE = '/content/drive/MyDrive/math_distill'
os.makedirs(BASE, exist_ok=True)

TEACHER_ID = 'Qwen/Qwen2.5-Math-1.5B-Instruct'
STUDENT_ID = 'Qwen/Qwen2.5-0.5B'
TEACHER_DIR = f'{BASE}/teacher'
STUDENT_DIR = f'{BASE}/student_base'
OUT_DIR = f'{BASE}/student_distilled'

N_TRAIN_Q = 1500   # تعداد سوال‌هایی که معلم برایشان حل تولید می‌کند
N_TEST = 100       # تعداد تست
EPOCHS = 2
BATCH = 2
ACCUM = 8
LR = 2e-5
MAX_NEW = 320
MAXLEN = 512
SEED = 42

random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'GPU فعال نیست: Runtime > Change runtime type > T4 GPU'
print(torch.cuda.get_device_name(0))

## ۱) دانلود مدل‌ها در Drive

In [ ]:
from huggingface_hub import snapshot_download
for rid, d in [(TEACHER_ID, TEACHER_DIR), (STUDENT_ID, STUDENT_DIR)]:
    if not os.path.exists(f'{d}/config.json'):
        snapshot_download(rid, local_dir=d, allow_patterns=['*.json', '*.safetensors', '*.txt'])
print(sorted(os.listdir(TEACHER_DIR)))
print(sorted(os.listdir(STUDENT_DIR)))

## ۲) داده و توابع کمکی

In [ ]:
from datasets import load_dataset

ds = load_dataset('openai/gsm8k', 'main')
def gold(a): return a.split('####')[-1].strip().replace(',', '')

train = [{'q': x['question'], 'gold': gold(x['answer'])} for x in ds['train']]
test_all = [{'q': x['question'], 'gold': gold(x['answer'])} for x in ds['test']]
random.Random(SEED).shuffle(train)
train = train[:N_TRAIN_Q]
test = test_all[:N_TEST]   # ۱۰۰ تست ثابت، جدا از آموزش

def boxed(text):
    i = text.rfind('\\boxed{')
    if i < 0:
        return None
    i += 7
    depth, j = 1, i
    while j < len(text) and depth:
        depth += (text[j] == '{') - (text[j] == '}')
        j += 1
    return text[i:j-1] if depth == 0 else None

NUM = re.compile(r'-?\d[\d,]*\.?\d*')

def first_num(s):
    m = NUM.findall(s.replace('$', ''))
    return float(m[0].replace(',', '').rstrip('.')) if m else None

def extract(text):
    b = boxed(text)
    if b is not None:
        return first_num(b)
    m = NUM.findall(text.replace('$', ''))
    return float(m[-1].replace(',', '').rstrip('.')) if m else None

def correct(pred, g):
    return pred is not None and abs(pred - float(g)) < 1e-4

def teacher_prompt(tok, q):
    msgs = [{'role': 'system', 'content': 'Please reason step by step, and put your final answer within \\boxed{}.'},
            {'role': 'user', 'content': q}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def student_prompt(q):
    return f'Question: {q}\nSolution (reason step by step, put the final answer in \\boxed{{}}):\n'

@torch.no_grad()
def generate(model, tok, prompts, bs=32, max_new=MAX_NEW):
    tok.padding_side = 'left'
    outs = []
    for i in range(0, len(prompts), bs):
        enc = tok(prompts[i:i+bs], return_tensors='pt', padding=True).to('cuda')
        g = model.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(g[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
        print(f'{len(outs)}/{len(prompts)}', end='\r')
    return outs

def evaluate(model, tok, items, prompt_fn, name, cut=False):
    outs = generate(model, tok, [prompt_fn(x['q']) for x in items])
    rows = []
    for x, o in zip(items, outs):
        if cut:
            o = o.split('\nQuestion:')[0]
        p = extract(o)
        rows.append({'q': x['q'], 'gold': x['gold'], 'pred': p, 'ok': bool(correct(p, x['gold'])), 'output': o})
    n_ok = sum(r['ok'] for r in rows)
    print(f'\n{name}: {n_ok}/{len(rows)} = {n_ok/len(rows):.1%}')
    return rows, n_ok / len(rows)

## ۳) معلم: ارزیابی و تولید داده‌ٔ آموزشی

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

ttok = AutoTokenizer.from_pretrained(TEACHER_DIR)
teacher = AutoModelForCausalLM.from_pretrained(TEACHER_DIR, torch_dtype=torch.float16).cuda().eval()

t_rows, t_acc = evaluate(teacher, ttok, test, lambda q: teacher_prompt(ttok, q), 'TEACHER on 100 tests')

DATA_PATH = f'{BASE}/distill_data.jsonl'
if not os.path.exists(DATA_PATH):
    outs = generate(teacher, ttok, [teacher_prompt(ttok, x['q']) for x in train], bs=48)
    kept = [{'q': x['q'], 'a': o.strip()} for x, o in zip(train, outs)
            if 'boxed' in o and correct(extract(o), x['gold'])]
    with open(DATA_PATH, 'w') as f:
        for r in kept:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

data = [json.loads(l) for l in open(DATA_PATH)]
print(len(data), 'verified teacher solutions out of', N_TRAIN_Q)

del teacher
torch.cuda.empty_cache()

## ۴) دانش‌آموز: ارزیابی قبل از آموزش

In [ ]:
stok = AutoTokenizer.from_pretrained(STUDENT_DIR)

samples = [d['q'] for d in data[:50]] + [d['a'] for d in data[:50]]
assert all(ttok(t).input_ids == stok(t).input_ids for t in samples), 'tokenizer های معلم و دانش‌آموز یکسان نیستند'

student = AutoModelForCausalLM.from_pretrained(STUDENT_DIR, torch_dtype=torch.float16).cuda().eval()
b_rows, b_acc = evaluate(student, stok, test, student_prompt, 'STUDENT BEFORE on 100 tests', cut=True)
del student
torch.cuda.empty_cache()

## ۵) آموزش دانش‌آموز روی حل‌های معلم

In [ ]:
student = AutoModelForCausalLM.from_pretrained(STUDENT_DIR, torch_dtype=torch.float32).cuda()
student.gradient_checkpointing_enable()
student.config.use_cache = False
student.train()

def encode(d):
    p = stok(student_prompt(d['q'])).input_ids
    a = stok(d['a']).input_ids + [stok.eos_token_id]
    return (p + a)[:MAXLEN], ([-100] * len(p) + a)[:MAXLEN]

examples = [encode(d) for d in data]

def batches(ex, bs):
    idx = list(range(len(ex)))
    random.shuffle(idx)
    for i in range(0, len(idx), bs):
        b = [ex[j] for j in idx[i:i+bs]]
        L = max(len(x[0]) for x in b)
        pad = stok.pad_token_id
        ids = torch.tensor([x[0] + [pad] * (L - len(x[0])) for x in b])
        lab = torch.tensor([x[1] + [-100] * (L - len(x[1])) for x in b])
        att = torch.tensor([[1] * len(x[0]) + [0] * (L - len(x[0])) for x in b])
        yield ids.cuda(), lab.cuda(), att.cuda()

opt = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=0.0)
scaler = torch.cuda.amp.GradScaler()
steps_total = EPOCHS * math.ceil(len(examples) / (BATCH * ACCUM))
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: min(1.0, (s + 1) / 10) * max(0.05, 1 - s / steps_total))

step, t0 = 0, time.time()
for ep in range(EPOCHS):
    for k, (ids, lab, att) in enumerate(batches(examples, BATCH)):
        with torch.autocast('cuda', dtype=torch.float16):
            loss = student(input_ids=ids, attention_mask=att, labels=lab).loss / ACCUM
        scaler.scale(loss).backward()
        if (k + 1) % ACCUM == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            scaler.step(opt); scaler.update()
            opt.zero_grad(set_to_none=True); sched.step(); step += 1
            if step % 10 == 0:
                print(f'epoch {ep} step {step}/{steps_total} loss {loss.item() * ACCUM:.3f} ({time.time() - t0:.0f}s)')

student.gradient_checkpointing_disable()
student.config.use_cache = True
student = student.half().eval()
student.save_pretrained(OUT_DIR, safe_serialization=True)
stok.save_pretrained(OUT_DIR)
print('saved to', OUT_DIR)

## ۶) ۱۰۰ تست بعد از آموزش و گزارش نهایی

In [ ]:
a_rows, a_acc = evaluate(student, stok, test, student_prompt, 'STUDENT AFTER on 100 tests', cut=True)

report = {
    'teacher': TEACHER_ID, 'student': STUDENT_ID,
    'n_test': N_TEST, 'train_examples': len(data), 'epochs': EPOCHS,
    'accuracy': {'teacher': t_acc, 'student_before': b_acc, 'student_after': a_acc},
    'details': {'teacher': t_rows, 'student_before': b_rows, 'student_after': a_rows},
}
with open(f'{BASE}/report_100_tests.json', 'w') as f:
    json.dump(report, f, ensure_ascii=False, indent=1)

print('\n===== RESULT (100 GSM8K test problems) =====')
print(f'Teacher         : {t_acc:.0%}')
print(f'Student before  : {b_acc:.0%}')
print(f'Student after   : {a_acc:.0%}')
print('report saved:', f'{BASE}/report_100_tests.json')

print('\n--- 3 failed examples (student after) ---')
for r in [r for r in a_rows if not r['ok']][:3]:
    print('Q:', r['q'][:200]); print('gold:', r['gold'], '| pred:', r['pred']); print(r['output'][:300]); print('---')

## اختیاری: تبدیل مدل نهایی به GGUF
سلول زیر را فقط در صورت نیاز از حالت کامنت دربیاور.

In [ ]:
# !git clone -q https://github.com/ggml-org/llama.cpp /content/llama.cpp
# !pip -q install -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
# !python /content/llama.cpp/convert_hf_to_gguf.py {OUT_DIR} --outfile {BASE}/student_distilled-f16.gguf --outtype f16